
# MPAL Manual Validation Notebook

This notebook loads the synthetic PHQ-9 and GAD-7 samples in `examples/manual_validation_*.csv`, scores each file via `psylab.score_responses`, and checks totals/severity labels against the manual scoring reference.


## What this validation shows

- Purpose: confirm MPAL scoring matches a hand-scored reference for PHQ-9 and GAD-7.
- How to read the tables: each row pairs the manual total/severity with the MPAL total/severity; matches mean the engine reproduces hand scoring.
- How to read the plots: bars compare severity category counts (Manual vs MPAL) so reviewers can see categories align at a glance.
- Takeaway: matching totals + matching labels demonstrate MPAL’s scoring logic aligns with the published cutoffs.




## How to use this notebook

1. Run the cells from top to bottom after installing MPAL (`pip install -e .[dev]`).
2. Inspect the comparison tables to confirm manual vs MPAL totals match.
3. Review the severity distribution figure to make sure MPAL reproduces the manual bands.
4. Replace the synthetic CSVs with pilot data (as permitted) to extend the validation study.


In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    def display(obj):  # type: ignore
        print(obj)

from psylab import load_instrument_spec, score_responses


plt.style.use('ggplot')


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'psylab').exists():
            return candidate
    raise RuntimeError(f"Unable to locate the MPAL repo root starting from {start}")


NOTEBOOK_CWD = Path().resolve()
PROJECT_ROOT = find_repo_root(NOTEBOOK_CWD)
DATA_DIR = PROJECT_ROOT / 'examples'
VALIDATION_DIR = PROJECT_ROOT / 'docs' / 'validation'

print(f"Notebook working directory: {NOTEBOOK_CWD}")
print(f"Project root resolved to: {PROJECT_ROOT}")
print(f"Loading validation CSVs from: {DATA_DIR}")


In [ ]:

validation_sources = {
    'phq9': DATA_DIR / 'manual_validation_phq9.csv',
    'gad7': DATA_DIR / 'manual_validation_gad7.csv',
}


def load_and_compare(instrument_id: str, csv_path: Path) -> dict:
    spec = load_instrument_spec(instrument_id)
    manual = pd.read_csv(csv_path)
    scored = score_responses(spec, manual)

    merged = manual.merge(
        scored,
        left_on='participant_id',
        right_on='respondent_id',
        how='inner',
        suffixes=('_manual', '_mpal'),
    )
    merged['score_diff'] = merged['total_score'] - merged['manual_total']
    merged['severity_match'] = merged['manual_severity'] == merged['severity']

    severity_thresholds = (
        (spec.get('scoring') or {})
        .get('interpretation', {})
        .get('severity_thresholds', [])
    )
    order = [band.get('label') for band in severity_thresholds if band.get('label')]

    return {
        'instrument': spec['instrument']['name'],
        'spec': spec,
        'data': merged,
        'severity_order': order,
    }


comparison_results = {
    instrument_id: load_and_compare(instrument_id, csv_path)
    for instrument_id, csv_path in validation_sources.items()
}

for info in comparison_results.values():
    print(f"Loaded {info['instrument']} validation sample with {len(info['data'])} respondents")


In [ ]:

preview_columns = [
    'participant_id',
    'manual_total',
    'total_score',
    'score_diff',
    'manual_severity',
    'severity',
    'severity_match',
]

for instrument_id, info in comparison_results.items():
    display_title = f"
{info['instrument']} manual vs MPAL comparison"
    print(display_title)
    display(
        info['data'][preview_columns].rename(
            columns={
                'total_score': 'mpal_total',
                'severity': 'mpal_severity',
            }
        )
    )


In [ ]:

summary_rows = []
for instrument_id, info in comparison_results.items():
    diff_ok = (info['data']['score_diff'].abs() < 1e-9).all()
    severity_ok = info['data']['severity_match'].all()
    summary_rows.append({
        'instrument': info['instrument'],
        'n': len(info['data']),
        'score_match': bool(diff_ok),
        'severity_match': bool(severity_ok),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if not summary_df['score_match'].all() or not summary_df['severity_match'].all():
    raise AssertionError('Manual vs MPAL parity check failed.')


In [ ]:

plot_rows = []
for instrument_id, info in comparison_results.items():
    df = info['data']
    order = info['severity_order'] or sorted(
        set(df['manual_severity']).union(set(df['severity']))
    )
    for source, column in [('Manual', 'manual_severity'), ('MPAL', 'severity')]:
        counts = (
            df[column]
            .value_counts()
            .reindex(order, fill_value=0)
            .reset_index()
            .rename(columns={'index': 'severity', column: 'count'})
        )
        counts['source'] = source
        counts['instrument'] = info['instrument']
        plot_rows.append(counts)

plot_df = pd.concat(plot_rows, ignore_index=True)

fig, axes = plt.subplots(1, len(comparison_results), figsize=(6 * len(comparison_results), 4), sharey=True)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, (instrument_id, info) in zip(axes, comparison_results.items()):
    name = info['instrument']
    order = info['severity_order'] or plot_df[plot_df['instrument'] == name]['severity'].unique().tolist()
    x = np.arange(len(order))
    width = 0.35

    manual_vals = (
        plot_df[(plot_df['instrument'] == name) & (plot_df['source'] == 'Manual')]
        .set_index('severity')
        .reindex(order, fill_value=0)['count']
        .to_numpy()
    )
    mpal_vals = (
        plot_df[(plot_df['instrument'] == name) & (plot_df['source'] == 'MPAL')]
        .set_index('severity')
        .reindex(order, fill_value=0)['count']
        .to_numpy()
    )

    ax.bar(x - width / 2, manual_vals, width=width, label='Manual')
    ax.bar(x + width / 2, mpal_vals, width=width, label='MPAL')
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=45, ha='right')
    ax.set_title(name)
    ax.set_ylabel('Respondents')
    ax.legend()

fig.suptitle('Severity distribution comparison (Manual vs MPAL)')
fig.tight_layout()



## Next steps

- Swap in additional CSVs (or append rows) to expand the validation sample before sharing with collaborators.
- Export the notebook as HTML/PDF when submitting artifacts to admissions reviewers or OSF.
- Capture any discrepancies in `LAUNCHPAD/next-actions.md` so they roll into the next sprint.
